# Wan 2.1 · Text to Video with Sound

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/wan2.1-t4-colab/blob/main/wan2_1_t4_colab_free_inference.ipynb)

Write a prompt, get a 2–5 second video with a matching sound track — on the **free Colab T4 GPU**.

**How to use**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`  *(first run downloads ~33 GB of weights: 5–15 min)*
3. Open the public `gradio.live` link printed by step 4.

<sub>Video: Wan 2.1 T2V-1.3B (4-bit NF4, UniPC). Sound: AudioLDM 2. Code: <a href="https://github.com/rivas-vega/wan2.1-t4-colab">github.com/rivas-vega/wan2.1-t4-colab</a></sub>

In [ ]:
# @title **1 · Setup** — check GPU and install (~1 min)
import os, sys, subprocess

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return r.stdout

try:
    gpu = _run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).strip()
    print(f"GPU  {gpu}")
except Exception:
    raise SystemExit("No GPU found. Go to Runtime → Change runtime type → T4 GPU, then run again.")

if os.path.isfile("pipeline_utils.py"):
    REPO_DIR = os.getcwd()
elif os.path.isdir("/content/wan2.1-t4-colab"):
    REPO_DIR = "/content/wan2.1-t4-colab"
    try:
        _run(["git", "-C", REPO_DIR, "pull", "-q"])
    except Exception:
        pass
else:
    REPO_DIR = "/content/wan2.1-t4-colab"
    _run(["git", "clone", "-q", "--depth", "1", "https://github.com/rivas-vega/wan2.1-t4-colab.git", REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

_run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # Colab's torchao breaks diffusers imports
_run([sys.executable, "-m", "pip", "install", "-q", "-U",
      "diffusers>=0.33.0", "transformers>=4.48.0", "accelerate>=0.34.0", "bitsandbytes",
      "gradio", "imageio[ffmpeg]", "ftfy", "sentencepiece", "protobuf", "scipy", "psutil"])
print("Ready ✓")

In [ ]:
# @title **2 · Hugging Face token** *(optional — 3–5× faster first download)*
# @markdown Free read token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). Leave empty to skip.
HF_TOKEN = ""  # @param {type:"string"}

import os
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
except Exception:
    pass

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("Token set ✓  fast downloads enabled")
else:
    print("No token — using anonymous downloads")

In [ ]:
# @title **3 · Load models** — first run 5–15 min, then cached
ENABLE_SOUND = True  # @param {type:"boolean"}

import os, warnings, psutil
warnings.filterwarnings("ignore")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")

from diffusers.utils import logging as _dl
from transformers.utils import logging as _tl
_dl.set_verbosity_error(); _tl.set_verbosity_error()

from pipeline_utils import load_audio_pipeline, load_wan_t4_pipeline, clear_cuda_cache

# Sound first: its loading peak happens while host RAM is still empty, then it lives on the GPU.
audio_pipe = None
if ENABLE_SOUND:
    print("Loading sound model (AudioLDM 2)…")
    try:
        audio_pipe = load_audio_pipeline()
    except Exception as e:
        print(f"Sound disabled ({type(e).__name__}: {e}). Videos will be silent.")

print("Loading video model (Wan 2.1, 4-bit)…")
pipe = load_wan_t4_pipeline(quantization="4bit")
clear_cuda_cache()

ram = (psutil.virtual_memory().total - psutil.virtual_memory().available) / 1e9
print(f"\nReady ✓  sound: {'on' if audio_pipe else 'off'} · host RAM in use: {ram:.1f} GB")

In [ ]:
# @title **4 · Open the app** — click the public link below
from app import build_demo, launch
demo = build_demo(pipe, audio_pipe, output_dir="/content/outputs")
launch(demo, share=True, debug=False)

In [ ]:
# @title **5 · Or generate here** *(optional)*
PROMPT = "A red vintage sports car drifting around a mountain hairpin at sunset, tire smoke, cinematic tracking shot"  # @param {type:"string"}
LENGTH = "3s"  # @param ["2s", "3s", "5s"]

from IPython.display import Video, display
from pipeline_utils import create_clip

path, seed, secs, sound = create_clip(pipe, audio_pipe, PROMPT, LENGTH, output_dir="/content/outputs")
print(f"{secs / 60:.1f} min · seed {seed} · {'with sound' if sound else 'silent'}")
display(Video(path, embed=True, width=640, html_attributes="controls autoplay loop"))